<a href="https://colab.research.google.com/github/lexinejazly-asuncion/Sign-Language-Classification/blob/main/Sign_Language_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas
!pip install pillow
!pip install kaggle
!pip install scikit-learn
!pip install torchvision

You should consider upgrading via the '/Users/richa.vakharia/Desktop/Sign-Language-Classification/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/richa.vakharia/Desktop/Sign-Language-Classification/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/richa.vakharia/Desktop/Sign-Language-Classification/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/richa.vakharia/Desktop/Sign-Language-Classification/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/richa.vakharia/Desktop/Sign-Language-Classification/venv/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import sys
!{sys.executable} -m pip install --upgrade pandas numexpr bottleneck
!{sys.executable} -m pip install "numpy<2"

     |████████████████████████████████| 145 kB 10.3 MB/s eta 0:00:01
     |████████████████████████████████| 104 kB 11.7 MB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Created wheel for bottleneck: filename=bottleneck-1.5.0-cp39-cp39-macosx_10_9_x86_64.whl size=108194 sha256=4caddf6e71940692674649f75b6ab88b9426b553dd1e6333d1ccc83d6ce9c2a6
  Stored in directory: /Users/richa.vakharia/Library/Caches/pip/wheels/4c/24/90/db649604e8e5b0b2b508c002619c2dbc06b0526c7e1926e564
Successfully built bottleneck
  Attempting uninstall: numexpr
    Found existing installation: numexpr 2.8.1
    Uninstalling numexpr-2.8.1:
      Successfully uninstalled numexpr-2.8.1
  Attempting uninstall: bottleneck
    Found existing installation: Bottleneck 1.3.4
    Uninstalling Bottleneck-1.3.4:
      Successfully uninstalled Bottleneck-1.3.4


In [3]:
!kaggle datasets download -d signnteam/asl-sign-language-pictures-minus-j-z -p dataset #download dataset

import zipfile

with zipfile.ZipFile("dataset/asl-sign-language-pictures-minus-j-z.zip", "r") as zip_ref: #extract from zip file
    zip_ref.extractall("dataset")

Dataset URL: https://www.kaggle.com/datasets/signnteam/asl-sign-language-pictures-minus-j-z
License(s): CC-BY-NC-SA-4.0
asl-sign-language-pictures-minus-j-z.zip: Skipping, found more recently modified local copy (use --force to force download)


## **EDA**

In [4]:
import pandas as pd
from pathlib import Path

In [5]:
data_path = Path("dataset/SigNN Character Database")

paths = [path.parts[-2:] for path in data_path.rglob("*.*")] #gives the class and image file name

sl_df = pd.DataFrame(data=paths, columns=["Class","Images"]) #create column names for dataframe
sl_df = sl_df.sort_values("Class",ascending=True)
sl_df.reset_index(drop=True, inplace=True)
sl_df.head()

,Class,Images
0,A,314.jpg
1,A,536.jpg
2,A,250.jpg
3,A,278.jpg
4,A,279.jpg


In [6]:
print(f"Total images in dataset: {len(sl_df)}")
print(f"Total classes: {sl_df['Class'].nunique()}")
print("\nTotal images per class:")
print(sl_df["Class"].value_counts().to_dict())

Total images in dataset: 8442
Total classes: 24

Total images per class:
{'B': 541, 'A': 539, 'E': 498, 'F': 420, 'C': 387, 'D': 379, 'O': 374, 'H': 364, 'I': 360, 'W': 347, 'L': 346, 'G': 345, 'V': 337, 'K': 319, 'Y': 318, 'S': 314, 'X': 310, 'T': 301, 'N': 293, 'R': 291, 'U': 286, 'M': 277, 'Q': 275, 'P': 221}


In [7]:
from PIL import Image
import os

#finds dimensions of each image
def get_img_dim(x):
  img_path = os.path.join(data_path, x["Class"], x["Images"])
  with Image.open(img_path) as img:
    return img.size

#creates two new columns in df: width and height
sl_df[["Width", "Height"]] = sl_df.apply(get_img_dim, axis=1, result_type="expand")
sl_df.head()

,Class,Images,Width,Height
0,A,314.jpg,640,480
1,A,536.jpg,1920,1920
2,A,250.jpg,640,480
3,A,278.jpg,640,480
4,A,279.jpg,640,480


In [8]:
#finds the different dimensions of each class
img_sizes = sl_df.groupby('Class').apply(
    lambda x: x[['Width', 'Height']].drop_duplicates().apply(
        lambda x: f"{int(x['Width'])}x{int(x['Height'])}", axis=1).tolist(), include_groups=False
)

print(f"Image sizes per class: {img_sizes}")


Image sizes per class: Class
A              [640x480, 1920x1920]
B              [1920x1920, 640x480]
C              [640x480, 1920x1920]
D              [640x480, 1920x1920]
E              [640x480, 1920x1920]
F              [640x480, 1920x1920]
G              [640x480, 1920x1920]
H              [640x480, 1920x1920]
I              [1920x1920, 640x480]
K              [640x480, 1920x1920]
L              [640x480, 1920x1920]
M              [640x480, 1920x1920]
N    [640x480, 1920x1920, 1280x720]
O              [640x480, 1920x1920]
P              [640x480, 1920x1920]
Q    [640x480, 1280x720, 1920x1920]
R              [1920x1920, 640x480]
S    [640x480, 1920x1920, 1280x720]
T              [640x480, 1920x1920]
U              [640x480, 1920x1920]
V    [640x480, 1920x1920, 1280x720]
W    [640x480, 1280x720, 1920x1920]
X              [640x480, 1920x1920]
Y              [640x480, 1920x1920]
dtype: object


## **Preprocessing**

In [9]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

/Users/richa.vakharia/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [19]:
train, test = train_test_split(sl_df, test_size=0.2, stratify=sl_df["Class"], random_state=42)

train_set = train.groupby("Class").sample(n=150, random_state=42).reset_index(drop=True)

In [20]:
class ASLDataset(Dataset):
    def __init__(self, df, data_path, transform=None):
        self.df = df
        self.data_path = data_path
        self.transform = transform
        self.label_map = {label: i for i, label in enumerate(sorted(df['Class'].unique()))} #for label encoding
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.data_path, row["Class"], row["Images"])
        
        image = Image.open(img_path).convert("RGB")
        label = self.label_map[row["Class"]] #labels represented as ints
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

preprocess_pipeline = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [21]:
#create the dataset object using the train_set (the 150 samples per class)
train_dataset = ASLDataset(train_set, data_path, transform=preprocess_pipeline)

#apply preprocessing
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [1]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)
print(labels[:10])

NameError: name 'train_loader' is not defined

## **Training**